In [ ]:
# 05 – Feature Engineering (Bowling)

Purpose:
- Compute bowler-level performance metrics
- Compute phase-wise bowling workload & control
- Create final bowling metrics dataset

Input:
- data/processed/deliveries_cleaned.csv

Output:
- data/processed/bowling_metrics.csv

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import *
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(PROCESSED_DIR / "deliveries_cleaned.csv")
print(df.columns)
df.head()

Index(['match_id', 'innings', 'batting_team', 'over', 'ball', 'batsman',
       'bowler', 'non_striker', 'runs_batsman', 'runs_extras', 'runs_total',
       'extras_type', 'player_dismissed', 'dismissal_kind', 'phase'],
      dtype='str')


,match_id,innings,batting_team,over,ball,batsman,bowler,non_striker,runs_batsman,runs_extras,runs_total,extras_type,player_dismissed,dismissal_kind,phase
0,1,1st innings,Sunrisers Hyderabad,0,1,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
1,1,1st innings,Sunrisers Hyderabad,0,2,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
2,1,1st innings,Sunrisers Hyderabad,0,3,DA Warner,TS Mills,S Dhawan,4,0,4,NaN,NaN,NaN,Powerplay
3,1,1st innings,Sunrisers Hyderabad,0,4,DA Warner,TS Mills,S Dhawan,0,0,0,NaN,NaN,NaN,Powerplay
4,1,1st innings,Sunrisers Hyderabad,0,5,DA Warner,TS Mills,S Dhawan,0,2,2,wides,NaN,NaN,Powerplay


In [5]:
df = pd.read_csv(PROCESSED_DIR / "deliveries_cleaned.csv")


df = df[~df["extras_type"].isin(["wides", "noballs"])].copy()



df["balls_faced"] = 1

In [7]:
df["balls_bowled"] = 1

bowling_basic = (
    df.groupby("bowler")
      .agg(
          runs_conceded=("runs_total", "sum"),
          balls=("balls_bowled", "sum"),
          wickets=(
              "dismissal_kind",
              lambda x: ((x.notna()) & (x != "run out")).sum()
          )
      )
      .reset_index()
)

bowling_basic["overs"] = bowling_basic["balls"] / 6

bowling_basic["economy"] = (
    bowling_basic["runs_conceded"] / bowling_basic["overs"]
)

In [8]:
phase_bowling = (
    df.groupby(["bowler", "phase"])
      .agg(
          runs=("runs_total", "sum"),
          balls=("balls_bowled", "sum")
      )
      .reset_index()
)

phase_bowling["overs"] = phase_bowling["balls"] / 6
phase_bowling["economy"] = (
    phase_bowling["runs"] / phase_bowling["overs"]
)

phase_bowling.head()

,bowler,phase,runs,balls,overs,economy
0,A Ashish Reddy,Death,164,101,16.833333,9.742574
1,A Ashish Reddy,Middle,204,155,25.833333,7.896774
2,A Ashish Reddy,Powerplay,20,6,1.000000,20.000000
3,A Badoni,Death,19,16,2.666667,7.125000
4,A Badoni,Middle,31,18,3.000000,10.333333


In [9]:
phase_economy = (
    phase_bowling
    .pivot(index="bowler", columns="phase", values="economy")
    .reset_index()
)

In [10]:
phase_balls = (
    df.groupby(["bowler", "phase"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

phase_balls.head()

phase,bowler,Death,Middle,Powerplay
0,A Ashish Reddy,101,155,6
1,A Badoni,16,18,1
2,A Chandila,6,84,144
3,A Choudhary,24,41,36
4,A Dananjaya,0,18,6


In [17]:
bowling_metrics = (
    bowling_basic
    .merge(phase_economy, on="bowler", how="left")
    .merge(phase_balls, on="bowler", how="left")
)

bowling_metrics.head()

,bowler,runs_conceded,balls,wickets,overs,economy,Death_x,Middle_x,Powerplay_x,Death_y,Middle_y,Powerplay_y
0,A Ashish Reddy,388,262,18,43.666667,8.885496,9.742574,7.896774,20.000000,101,155,6
1,A Badoni,50,35,4,5.833333,8.571429,7.125000,10.333333,0.000000,16,18,1
2,A Chandila,245,234,11,39.000000,6.282051,6.000000,7.142857,5.791667,6,84,144
3,A Choudhary,137,101,5,16.833333,8.138614,9.250000,8.048780,7.500000,24,41,36
4,A Dananjaya,46,24,0,4.000000,11.500000,NaN,11.333333,12.000000,0,18,6


In [15]:
bowling_metrics = bowling_metrics.rename(columns={
    # Economy columns
    "Powerplay_x": "powerplay_economy",
    "Middle_x": "middle_overs_economy",
    "Death_x": "death_overs_economy",

    # Balls bowled columns
    "Powerplay_y": "powerplay_balls_bowled",
    "Middle_y": "middle_overs_balls_bowled",
    "Death_y": "death_overs_balls_bowled"
})

In [16]:
bowling_metrics["total_balls_bowled"] = (
    bowling_metrics["Powerplay"]
    + bowling_metrics["Middle"]
    + bowling_metrics["Death"]
)

In [18]:
bowling_metrics.to_csv(
    PROCESSED_DIR / "bowling_metrics.csv",
    index=False
)

print("bowling_metrics.csv created successfully")

bowling_metrics.csv created successfully
